In [1]:
import pandas as pd

df = pd.read_csv('train.csv')
df_geo = pd.read_csv('devices.csv')

df = df.merge(df_geo, on='deviceId', how='left')

In [2]:
df.head()

,deviceId,timedate,period,t1,t2,t3,t4,t5,t6,t7,...,t10,t11,t12,t13,x1,x2,x3,deviceType,latitude,longitude
0,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024-10-01 00:00:00,train,0.29,0.05,0.0,0.43,0.47,0.45,0.2,...,0.21,0.21,0.07,0.07,0.0,0.0,8,19,50.0,18.3
1,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024-10-01 00:05:00,train,0.29,0.05,0.0,0.39,0.46,0.45,0.2,...,0.21,0.21,0.07,0.07,0.0,0.0,8,19,50.0,18.3
2,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024-10-01 00:10:00,train,0.29,0.05,0.0,0.38,0.46,0.45,0.2,...,0.21,0.21,0.07,0.07,0.0,0.0,8,19,50.0,18.3
3,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024-10-01 00:15:00,train,0.29,0.05,0.0,0.38,0.45,0.45,0.2,...,0.21,0.21,0.07,0.07,0.0,0.0,8,19,50.0,18.3
4,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024-10-01 00:20:00,train,0.29,0.05,0.0,0.37,0.45,0.45,0.2,...,0.21,0.21,0.07,0.07,0.0,0.0,8,19,50.0,18.3


In [3]:
import numpy as np

df['deviceId'] = df['deviceId'].astype('category')
df['deviceType'] = df['deviceType'].astype('category')

numeric_features = ['t1', 't2', 't3', 't4', 't5', 't6', 't7', 't8', 't9', 't10', 't11', 't12', 't13', 'x1', 'x3', 'latitude', 'longitude']

corr_matrix = df[numeric_features].corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

threshold = 0.90
to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > threshold)]

print(f"{to_drop}")

numeric_features_filtered = [f for f in numeric_features if f not in to_drop]

['t6', 't12', 't13']


In [ ]:
df['timedate'] = pd.to_datetime(df['timedate'])

df['year_month'] = df['timedate'].dt.to_period('h')

features = numeric_features_filtered

agg_funcs = {feat: ['mean', 'std', 'min', 'max'] for feat in features}

agg_funcs['x2'] = ['mean']

df_hourly = df.groupby(['deviceId', 'deviceType', 'year_month'], observed=True).agg(agg_funcs).reset_index()
df_hourly.columns = ['_'.join(col).strip() if col[1] else col[0] for col in df_hourly.columns.values]

df_hourly = df_hourly.dropna()

X = df_hourly.drop(columns=['deviceId', 'year_month', 'x2_mean'])
y = df_hourly['x2_mean']

In [ ]:
X.head()

,deviceType,t1_mean,t2_mean,t3_mean,t4_mean,t5_mean,t7_mean,t8_mean,t9_mean,t10_mean,t11_mean,x1_mean,x3_mean,latitude_mean,longitude_mean
0,19,0.290000,0.05,0.0,0.372500,0.446667,0.200000,0.499167,0.394167,0.210000,0.210000,0.0,8.0,50.0,18.3
1,19,0.290000,0.05,0.0,0.418333,0.464167,0.200000,0.514167,0.408333,0.210000,0.205833,0.0,8.0,50.0,18.3
2,19,0.290000,0.05,0.0,0.447500,0.477500,0.205833,0.530833,0.390000,0.210000,0.202500,0.0,8.0,50.0,18.3
3,19,0.286667,0.05,0.0,0.392500,0.459167,0.210000,0.506667,0.403333,0.210000,0.209167,0.0,8.0,50.0,18.3
4,19,0.280000,0.05,0.0,0.445000,0.473333,0.210000,0.521667,0.415000,0.208333,0.204167,0.0,8.0,50.0,18.3


In [ ]:
from flaml import AutoML

automl = AutoML()

settings = {
    "time_budget": 5000,
    "metric": 'mae',
    "task": 'regression',
    "estimator_list": ['xgboost', 'lgbm', "catboost"]
}

automl.fit(X_train=X, y_train=y, **settings)

[flaml.automl.logger: 03-15 09:24:59] {1752} INFO - task = regression
[flaml.automl.logger: 03-15 09:24:59] {1763} INFO - Evaluation method: holdout
[flaml.automl.logger: 03-15 09:25:02] {1862} INFO - Minimizing error metric: mae
[flaml.automl.logger: 03-15 09:25:02] {1979} INFO - List of ML learners in AutoML Run: ['xgboost', 'lgbm', 'catboost']
[flaml.automl.logger: 03-15 09:25:02] {2282} INFO - iteration 0, current learner xgboost
[flaml.automl.logger: 03-15 09:25:02] {2417} INFO - Estimated sufficient time budget=657085s. Estimated necessary time budget=723s.
[flaml.automl.logger: 03-15 09:25:02] {2466} INFO -  at 7.1s,	estimator xgboost's best error=0.1077,	best estimator xgboost's best error=0.1077
[flaml.automl.logger: 03-15 09:25:02] {2282} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 03-15 09:25:02] {2466} INFO -  at 7.3s,	estimator lgbm's best error=0.1077,	best estimator lgbm's best error=0.1077
[flaml.automl.logger: 03-15 09:25:02] {2282} INFO - iteration 

In [ ]:
import pandas as pd
df = None
df_hourly = None
X = None
y = None

In [ ]:
df_valid = pd.read_csv('valid.csv')
df_test = pd.read_csv('test.csv')

df_combined = pd.concat([df_valid, df_test], ignore_index=True)
df_combined = df_combined.merge(df_geo, on='deviceId', how='left')

df_combined['timedate'] = pd.to_datetime(df_combined['timedate'])
df_combined['year'] = df_combined['timedate'].dt.year
df_combined['month'] = df_combined['timedate'].dt.month
df_combined['day'] = df_combined['timedate'].dt.day
df_combined['hour'] = df_combined['timedate'].dt.hour

features = numeric_features_filtered
agg_funcs = {feat: ['mean', 'std', 'min', 'max'] for feat in features}

df_combined_hourly = df_combined.groupby(['deviceId', 'deviceType', 'year', 'month', 'day', 'hour']).agg(agg_funcs).reset_index()

df_combined_hourly.columns = [
    '_'.join(col).strip() if col[1] else col[0] 
    for col in df_combined_hourly.columns.values
]


In [ ]:


X_test = df_combined_hourly.drop(columns=['deviceId', 'year', 'month', 'day', 'hour'])

predictions = automl.predict(X_test)

df_combined_hourly['hourly_prediction'] = predictions

submission = df_combined_hourly.groupby(['deviceId', 'year', 'month'])['hourly_prediction'].mean().reset_index()

submission.rename(columns={'hourly_prediction': 'prediction'}, inplace=True)

submission.to_csv('submission.csv', index=False)